# ZINC Feasibility-Rate Hyperparameter Search

This notebook runs a small random search over ZINC generator hyperparameters and ranks fitted trials by the rate of feasible generated molecules.

This notebook trains a ZINC graph generator on a size-filtered subset, scores each fitted model by feasible generation rate, and runs a small random search over an explicitly typed hyperparameter space.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from conditional_node_field_graph_generator.notebooks import configure_notebook

globals().update(configure_notebook(require_nsppk=True, print_torch=False))

from conditional_node_field_graph_generator.extensions.demo import (
    build_graph_generator,
    build_zinc_dataset,
    sample_hyperparameter_configuration,
)
from abstractgraph_graphicalizer.chem import draw_molecules


## Dataset

Specify only the size range and number of molecules to keep the ZINC setup compact.

In [ ]:
NOTEBOOK_DATA_ROOT = REPO_ROOT / 'notebooks' / 'datasets'
ARTIFACT_ROOT = REPO_ROOT / '.artifacts' / 'zinc_hyperparameter_search'
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'

NUM_EXAMPLES = 500
MIN_SIZE = 4
MAX_SIZE = 10

graphs, zinc_metadata, corpus_manifest = build_zinc_dataset(
    dataset_dir=ZINC_DATA_ROOT,
    num_examples=NUM_EXAMPLES,
    min_size=MIN_SIZE,
    max_size=MAX_SIZE,
)

node_label_set = sorted({attrs.get('label') for graph in graphs for _, attrs in graph.nodes(data=True) if attrs.get('label') is not None}, key=str)
edge_label_set = sorted({attrs.get('label') for graph in graphs for _, _, attrs in graph.edges(data=True) if attrs.get('label') is not None}, key=str)

print(f"Corpus cache: {ZINC_DATA_ROOT}")
print(f"Available node counts: {corpus_manifest['node_counts'][:10]} ... {corpus_manifest['node_counts'][-10:]}")
print(f"Loaded graphs: {len(graphs)}")
print(f"Distinct node labels: {len(node_label_set)} | {node_label_set}")
print(f"Distinct edge labels: {len(edge_label_set)} | {edge_label_set}")
display(zinc_metadata.head())
draw_molecules(graphs[:7*4])


## Search Space

The search space is explicit: every tunable parameter has a range and a declared type. The feasible-rate score configuration is fixed for the whole experiment and is not part of the search space.

In [ ]:
N_TRIALS = 4
SCORE_N_SAMPLES = 16
SCORE_MAX_FEASIBILITY_ATTEMPTS = 10
SCORE_FEASIBILITY_CANDIDATES_PER_ATTEMPT = 16
SEARCH_RANDOM_STATE = 42
VERBOSE = 1

base_generator_kwargs = {
    'nbits': 11,
    'verbose': VERBOSE,
    'latent_embedding_dimension': 64,
    'number_of_transformer_layers': 2,
    'transformer_attention_head_count': 4,
    'transformer_dropout': 0.15,
    'learning_rate': 2e-4,
    'maximum_epochs': 80,
    'batch_size': 16,
    'total_steps': 80,
    'verbose_epoch_interval': 10,
    'enable_early_stopping': True,
    'early_stopping_monitor': 'val_total',
    'early_stopping_mode': 'min',
    'early_stopping_patience': 15,
    'early_stopping_min_delta': 5.0,
    'restore_best_checkpoint': True,
    'use_feasibility_filtering': True,
    'feasibility_failure_mode': 'return_partial',
    'decoder_enforce_connectivity': True,
    'decoder_n_jobs': 1,
}

search_space = {
    'lambda_degree_importance': {'type': 'real', 'low': 0.1, 'high': 4.0},
    'lambda_node_exist_importance': {'type': 'real', 'low': 0.1, 'high': 4.0},
    'lambda_node_count_importance': {'type': 'real', 'low': 0.0, 'high': 2.0},
    'lambda_node_label_importance': {'type': 'real', 'low': 0.1, 'high': 4.0},
    'lambda_edge_label_importance': {'type': 'real', 'low': 0.1, 'high': 4.0},
    'lambda_direct_edge_importance': {'type': 'real', 'low': 0.1, 'high': 4.0},
    'lambda_edge_count_importance': {'type': 'real', 'low': 0.0, 'high': 2.0},
    'lambda_degree_edge_consistency_importance': {'type': 'real', 'low': 0.0, 'high': 2.0},
    'lambda_auxiliary_edge_importance': {'type': 'real', 'low': 0.0, 'high': 2.0},
    'sampling_steps': {'type': 'int', 'low': 20, 'high': 120},
    'sampling_step_size': {'type': 'real', 'low': 0.01, 'high': 0.10},
    'sparse_supervision_mask_ratio': {'type': 'real', 'low': 0.0, 'high': 0.8},
    'degree_temperature': {'type': 'real', 'low': 0.7, 'high': 1.5},
}

display(pd.DataFrame(search_space).T)


## Trial Runner

The score is the candidate-level feasible rate returned by `graph_generator.score_feasible_rate(...)`. `max_feasibility_attempts`, `feasibility_candidates_per_attempt`, and `n_samples` are fixed experiment settings here, so the search only explores generator hyperparameters.

In [ ]:
def run_trial(trial_id, sampled_params):
    trial_root = ARTIFACT_ROOT / f'trial_{trial_id:03d}'
    trial_root.mkdir(parents=True, exist_ok=True)

    graph_generator = build_graph_generator(
        **base_generator_kwargs,
        **sampled_params,
        artifact_root=trial_root / 'artifacts',
        checkpoint_root=trial_root / 'checkpoints',
    )
    graph_generator.fit(graphs)

    score_info = graph_generator.score_feasible_rate(
        n_samples=SCORE_N_SAMPLES,
        max_feasibility_attempts=SCORE_MAX_FEASIBILITY_ATTEMPTS,
        feasibility_candidates_per_attempt=SCORE_FEASIBILITY_CANDIDATES_PER_ATTEMPT,
    )
    return {
        'trial_id': trial_id,
        **sampled_params,
        **score_info,
    }


## Random Search

In [ ]:
results = []
for trial_id in range(1, N_TRIALS + 1):
    sampled_params = sample_hyperparameter_configuration(
        search_space,
        random_state=SEARCH_RANDOM_STATE + trial_id,
    )
    print(f"\n=== Trial {trial_id}/{N_TRIALS} ===")
    print(sampled_params)
    trial_result = run_trial(trial_id, sampled_params)
    results.append(trial_result)
    print(
        f"score={trial_result['score']:.3f} | "
        f"fulfilled_rate={trial_result['fulfilled_rate']:.3f} | "
        f"accepted_slots={trial_result['accepted_slots']}/{trial_result['n_samples']}"
    )

results_df = pd.DataFrame(results).sort_values(['score', 'fulfilled_rate'], ascending=False).reset_index(drop=True)
display(results_df)


In [ ]:
best_row = results_df.iloc[0]
print('Best configuration:')
display(best_row.to_frame(name='value'))
